In [ ]:
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search
from google.genai import types

print("✅ ADK components imported successfully.")

In [ ]:
from google.genai import types

deterministic_config = types.GenerateContentConfig(
    temperature=0.0,
    top_p=1.0,
    seed=42
)

In [ ]:
from google.adk.agents import Agent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.runners import InMemoryRunner
from google.adk.tools import AgentTool, FunctionTool, google_search
from google.genai import types



def exit_loop():
    """Call this function ONLY when the examples are valid and verify agent approves as "True", indicating the examples matches the criteria and no more changes are needed."""
    return {"status": "approved", "message": " Exiting refinement loop."}

# 1. Initial Search Agent (The Draft Creator)
webSearch_agent = Agent(
    name="webSearch_agent",
    model="gemini-2.5-flash-lite",
    generate_content_config=deterministic_config,
    instruction="""You are a web Search agent.
    1. Refer to existing web dictionaries, wikipedia and lexical resources to identify the possible examples to match the given word sense.
    2. You need to return 2 illustrative examples for each given word sense from the list.
    3. Provide concise and accurate information based on the search results. 
    4. Do not add additional comments or information. Only sense ID, its meaning and 2 examples are expected. The results needs to be return in a proper JSON format.""",
    tools=[google_search],
    output_key="websearch_results", # Stores the initial draft
)


root_agent = SequentialAgent(
    name="StoryPipeline",
    sub_agents=[webSearch_agent],
)

runner = InMemoryRunner(agent=root_agent)



In [ ]:
from huggingface_hub import login
from datasets import load_dataset
import ast
from google.genai import types

# Load dataset from Hugging Face
dataset = load_dataset("dataset_name", split="test")

for line_num, sample in enumerate(dataset, 1):
  sentence = sample["text"]
  word = sample["word"]
  pos = sample["pos"]
  gold = sample["gold"]
  senses = sample["possible_meanings"]



  try:
    lst = ast.literal_eval(f"[{senses}]")
  except Exception:
    lst = senses

  # 1. Build the prompt (included {sentence} here)
  prompt = f"""You are given an ambiguous word: "{word}".
This word has multiple distinct senses: {lst}

Use internal tools and identify related examples illustrating all the meanings.
Return only the examples in JSON format. Do not add extra details or explanations."""

  response = await runner.run_debug(prompt)   

 

       
